<a href="https://colab.research.google.com/github/aniket-alt/unsloth/blob/main/2_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U unsloth transformers trl datasets accelerate peft bitsandbytes xformers gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [2]:
# ==== System setup (Colab/Kaggle) ====

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
BF16 = is_bfloat16_supported() # autocasts to bf16 if available

# Utility: tiny evaluation helper
def chat(model, tokenizer, user, system="You are a helpful assistant.", max_new_tokens=128):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    # 1. Tokenize. This returns a single PyTorch Tensor.
    inputs_tensor = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    )

    # 2. Move the tensor to the model's device
    inputs_on_device = inputs_tensor.to("cuda")

    from transformers import TextStreamer
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        # 3. Call generate.
        # We MUST pass `input_ids` as the first argument to avoid
        # a 'KeyError: key' bug in this version of Unsloth.
        _ = model.generate(
            input_ids = inputs_on_device, # <--- This must be the first argument
            streamer = streamer,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = 0.7
        )

print("Setup complete.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Setup complete.


In [3]:
# ==== B. LoRA finetune (Phi‑3.5 Mini 4‑bit) ====
MODEL = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"  # 4-bit, small VRAM
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
  # Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj","embed_tokens","lm_head"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.11.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [5]:
# Dataset: reuse the same mapping as A (small slice for speed)
ds = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")

def to_text(batch):
    texts = []
    for ins, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user = ins if not inp else f"{ins}\n\n{inp}"
        msgs = [
            {"role":"system","content":"You are a helpful assistant."},
            {"role":"user","content":user},
            {"role":"assistant","content":out},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))
    return {"text": texts}

train = ds.map(to_text, batched=True, remove_columns=ds.column_names)

from trl import SFTTrainer
from transformers import TrainingArguments

args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=300,               # quick run; prefer epochs for real runs
    learning_rate=2e-4,
    bf16=BF16, fp16=not BF16,
    logging_steps=10,
    save_steps=100,
    output_dir="lora-phi35-mini",
    optim="adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=True,
    args=args,
)
trainer.train()

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 2,000 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 226,885,632 of 4,047,965,184 (5.60% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.218800
20,1.105200
30,1.048800
40,0.946500
50,0.956300
60,0.889100
70,0.895000
80,0.928700
90,0.914100
100,0.897000


TrainOutput(global_step=300, training_loss=0.7911720784505208, metrics={'train_runtime': 3043.6807, 'train_samples_per_second': 1.577, 'train_steps_per_second': 0.099, 'total_flos': 3.0311137642254336e+16, 'train_loss': 0.7911720784505208, 'epoch': 2.4})

In [6]:
# # Test
# FastLanguageModel.for_inference(model)
# print("\n=== Test (LoRA) ===")
# chat(model, tokenizer, "Write a friendly 2‑line welcome message for new users.")

# 4) Test
from unsloth import FastLanguageModel
import torch

target_dtype = torch.bfloat16 if BF16 else torch.float16
print(f"Manually casting model to {target_dtype} for inference...")

model.to(target_dtype)

FastLanguageModel.for_inference(model)
print("\n=== Test ===")
chat(model, tokenizer, "Write a friendly 2‑line welcome message for new users.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Manually casting model to torch.float16 for inference...

=== Test ===
"Welcome to our community! We're thrilled to have you here and can't wait to get to know you better. Enjoy your journey with us!"


In [7]:
# ==== Minimal Gradio chat ====
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread
FastLanguageModel.for_inference(model)

def respond(message, history):
    msgs = []
    for u,a in history + [(message,"")]:
        msgs.append({"role":"user","content":u})
        msgs.append({"role":"assistant","content":a})
    msgs.pop()  # drop trailing empty assistant
    input_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    def run_gen(): model.generate(input_ids=input_ids, streamer=streamer, max_new_tokens=256, temperature=0.7, do_sample=True)
    Thread(target=run_gen).start()
    partial=""
    for token in streamer:
        partial += token
        yield partial

gr.ChatInterface(respond, title="Unsloth Chat UI").launch(share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>